# C12-classical-models — Practice p13 — Solution


Empty clusters are repaired deterministically from assignment-step distances before exact mean updates. Each trace value uses the repaired labels and their updated centroids.


In [ ]:
import numpy as np

X_p13 = np.array([[0.,0.],[0.,2.],[1.,1.],[8.,8.],[8.,10.],[9.,9.]], dtype=np.float64)
initial_centroids_p13 = np.array([[0.,0.],[0.,0.],[9.,9.]], dtype=np.float64)


def lloyd_kmeans(X, initial_centroids, max_iter=100, tol=1e-10):
    if not all(isinstance(a, np.ndarray) and a.dtype == np.float64 for a in (X, initial_centroids)):
        raise ValueError("X and centroids must be float64 arrays")
    if X.ndim != 2 or initial_centroids.ndim != 2 or X.shape[0] < 1 or X.shape[1] < 1:
        raise ValueError("invalid dimensions")
    if initial_centroids.shape[1] != X.shape[1] or not 1 <= initial_centroids.shape[0] <= X.shape[0]:
        raise ValueError("invalid centroid shape")
    if not np.isfinite(X).all() or not np.isfinite(initial_centroids).all():
        raise ValueError("inputs must be finite")
    if not isinstance(max_iter, (int, np.integer)) or isinstance(max_iter, bool) or max_iter <= 0:
        raise ValueError("max_iter must be positive")
    if not np.isscalar(tol) or not np.isfinite(tol) or tol < 0:
        raise ValueError("tol must be nonnegative")
    centroids = initial_centroids.copy()
    previous_labels = None
    trace = []
    k = centroids.shape[0]
    for _ in range(int(max_iter)):
        distances = ((X[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
        labels = np.argmin(distances, axis=1).astype(np.int64)
        assignment_distances = distances[np.arange(X.shape[0]), labels].copy()
        for empty in np.flatnonzero(np.bincount(labels, minlength=k) == 0):
            counts = np.bincount(labels, minlength=k)
            eligible = np.flatnonzero(counts[labels] >= 2)
            farthest = assignment_distances[eligible].max()
            row = int(eligible[np.flatnonzero(assignment_distances[eligible] == farthest)[0]])
            labels[row] = int(empty)
        # PLAN018_MUTATION_TARGET: C12-p13-centroid-update
        updated = np.vstack([X[labels == cluster].mean(axis=0) for cluster in range(k)]).astype(np.float64)
        objective = float(((X - updated[labels]) ** 2).sum())
        trace.append(objective)
        unchanged = previous_labels is not None and np.array_equal(labels, previous_labels)
        movement = float(np.linalg.norm(updated - centroids, axis=1).max())
        centroids = updated
        if unchanged or movement <= tol:
            break
        previous_labels = labels.copy()
    return labels, centroids, np.asarray(trace, dtype=np.float64)


labels_p13, centroids_p13, objective_trace_p13 = lloyd_kmeans(
    X_p13, initial_centroids_p13
)


### Answer check


In [ ]:
ATOL = 1e-10
RTOL = 1e-8
assert np.array_equal(labels_p13, [0, 1, 0, 2, 2, 2])
# PLAN018_ANSWER_CHECK: C12-p13-lloyd-update
assert np.allclose(centroids_p13, [[0.5,0.5],[0.0,2.0],[25.0/3.0,9.0]], atol=ATOL, rtol=RTOL)
assert np.allclose(objective_trace_p13, [11.0/3.0, 11.0/3.0], atol=ATOL, rtol=RTOL)
assert np.all(np.diff(objective_trace_p13) <= ATOL + RTOL * np.abs(objective_trace_p13[:-1]))
assert np.all(np.bincount(labels_p13, minlength=3) > 0)
